In [19]:
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env
import numpy as np

In [20]:
SIZE = 800
WIDTH = SIZE // 20
NB_BLOC = SIZE // WIDTH

In [21]:
class GridMaps(gym.Env):
    def __init__(self):
        super().__init__()
        self.wall = 0
        self.observation_space = spaces.Dict({
            'agent': spaces.Box(low=0, high=NB_BLOC-1, shape=(2, ), dtype=np.int32),
            'goal': spaces.Box(low=0, high=NB_BLOC-1, shape=(2, ), dtype=np.int32),
            'map': spaces.Box(low=0, high=20, shape=(NB_BLOC, NB_BLOC), dtype=np.int64)
        })
        # haut   0
        # bas    1
        # gauche 2
        # droite 3
        self.action_space = spaces.Discrete(4)
        self.position = np.array([0, 0], dtype=np.int32)
        self.goals = np.array([4, 3], dtype=np.int32)

    def reset(self, *, seed = None, options = None):
        super().reset(seed=seed, options=options)
        if options and 'map' in options:
            self.map = options['map']
        else:
            self.map = self.np_random.choice([0, 1], size=(NB_BLOC, NB_BLOC), p=[1 - self.wall, self.wall])
        if options and 'position' in options:
            self.position = options['position']
        else:
            self.position = self.np_random.integers(0, NB_BLOC, size=(2, ),dtype=np.int32)
        if options and 'goals' in options:
            self.goals = options['goals']
        else:
            self.goals = self.np_random.integers(0, NB_BLOC, size=(2, ),dtype=np.int32)
            while self.map[self.goals[0], self.goals[1]] == 1:
                self.goals = self.np_random.integers(0, NB_BLOC, size=(2, ),dtype=np.int32)
        self.map[self.position[0]][self.position[1]] = 0
        return {'agent': self.position.copy(),'goal': self.goals.copy(),'map': self.map.copy()}, {}

    def action_masks(self):
        move = np.ones(4, dtype=np.bool)
        x, y = self.position
        m = self.map
        move[3] = False if x+1 < NB_BLOC and m[x+1][y] == 1 else True
        move[2] = False if x-1 >= 0 and m[x-1][y] == 1 else True
        move[1] = False if y+1 < NB_BLOC and m[x][y+1] == 1 else True
        move[0] = False if y-1 >= 0 and m[x][y-1] == 1 else True
        return move

    def step(self, action):
        reward = -0.1
        x, y = self.position
        if action == 0:
            y -= 1
        if action == 1:
            y += 1
        if action == 2:
            x -= 1
        if action == 3:
            x += 1
        if x< 0 or x> NB_BLOC - 1 or y < 0 or y > NB_BLOC - 1 or self.map[x, y] == 1:
            reward -= 10
        else:
            self.position = np.array([x, y])
        terminated = np.array_equal(self.goals, self.position)
        if terminated:
            reward += 50
        return {'agent': self.position,'goal': self.goals,'map': self.map}, reward, terminated, False, {}

In [22]:
env = GridMaps()

obs, info = env.reset()

for i in range(2):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action=action)
    print(obs)
    print('='*20)

check_env(env=env)

{'agent': array([7, 9], dtype=int32), 'goal': array([17,  1], dtype=int32), 'map': array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 

In [23]:
from sb3_contrib import MaskablePPO

In [24]:
env = gym.wrappers.TimeLimit(env, max_episode_steps=400)

agent = MaskablePPO('MultiInputPolicy', env=env, verbose=1)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [ ]:
agent.learn(total_timesteps=10000)
env.unwrapped.wall = 0.1
agent.learn(total_timesteps=10000)
env.unwrapped.wall = 0.15
agent.learn(total_timesteps=10000)
env.unwrapped.wall = 0.2
agent.learn(total_timesteps=10000)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 332      |
|    ep_rew_mean     | -143     |
| time/              |          |
|    fps             | 665      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 363         |
|    ep_rew_mean          | -122        |
| time/                   |             |
|    fps                  | 594         |
|    iterations           | 2           |
|    time_elapsed         | 6           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011933681 |
|    clip_fraction        | 0.144       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | -0.0125     |
|    learning_rate        | 0.

In [ ]:
act = {
    'haut': 0,
    'bas': 1,
    'gauche': 2,
    'droite': 3
}

def get_action(number):
    for k,v in act.items():
        if v == number:
            return k, v

In [ ]:
import pygame

pygame.init()

WHITE_COLOR = (255, 255, 255)
PADDING = 0
FPS = 20

screen = pygame.display.set_mode((SIZE + PADDING, SIZE + PADDING))
time = pygame.time.Clock()

loop = True

obs, info = env.reset()
print(obs)
goal = obs['goal']
def create_grid(obs):
    m = obs['map']
    for i in range(len(m[0])):
        for j in range(len(m[1])):
            if m[i, j] == 0:
                pygame.draw.rect(screen,pygame.Color(WHITE_COLOR), pygame.Rect(i * WIDTH + PADDING, j * WIDTH +PADDING , WIDTH - PADDING, WIDTH - PADDING))
            elif m[i, j] == 1:
                pygame.draw.rect(screen,pygame.Color(0, 0, 0), pygame.Rect(i * WIDTH + PADDING, j * WIDTH +PADDING , WIDTH - PADDING, WIDTH - PADDING))

def draw_obj(x, y, color, decalage = 0):
    pygame.draw.circle(screen,pygame.Color(color), (x * WIDTH + WIDTH // 2 - decalage, y * WIDTH + WIDTH // 2), WIDTH // 2)

terminated = False
decalage = 0
while loop:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            loop = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_n:
                terminated = True
    screen.fill(color=WHITE_COLOR)
    if terminated:
        decalage = PADDING // 2 if PADDING > 0 else 10
        print(terminated)
        obs, info = env.reset()
        goal = obs['goal']
        terminated = False
        decalage = 0
    create_grid(obs=obs)
    draw_obj(obs['agent'][0],obs['agent'][1], (0, 0, 255), decalage=decalage)
    draw_obj(goal[0], goal[1], (200,0, 100), decalage=-decalage)
    pygame.display.update()
    time.tick(FPS)
    if terminated == False:
        mask = env.unwrapped.action_masks()
        action, state = agent.predict(obs, action_masks=mask)
        obs, reward, terminated, truncated, info = env.step(action=action)
pygame.quit()

{'agent': array([10,  8], dtype=int32), 'goal': array([13,  7], dtype=int32), 'map': array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0

KeyboardInterrupt: 